# GLMMadaptive Basics (Python)

**Author:** Dimitris Rizopoulos

This notebook is a Python translation of the R vignette *GLMMadaptive Basics*,
using the `glmmadaptive` Python package — a port of the R package of the same
name.  All models are fitted with **adaptive Gaussian quadrature** exactly as
in the R implementation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.special import expit        # equivalent to R's plogis()
from scipy.stats import chi2
import statsmodels.formula.api as smf

from glmmadaptive import MixedModel
from glmmadaptive.families import Binomial, Poisson
from glmmadaptive.results import MixModResults

---

## 1  Generalized Linear Mixed Models — Theory

### 1.1  Model Specification

Package **glmmadaptive** provides a suite of functions for fitting and
post-processing mixed effects models for grouped/clustered outcomes which have a
distribution other than a normal distribution.  In particular, let $y_i$ denote
a vector of grouped/clustered outcome for the $i$-th sample unit
($i = 1, \ldots, n$).  The conditional distribution of $y_i$ given a vector of
random effects $b_i$ is assumed to be a member of the extended exponential
family, with linear predictor given by
$$g\{E(y_i \mid b_i)\} = X_i \beta + Z_i b_i,$$
where $g(\cdot)$ denotes a monotonic link function, $X_i$ a design matrix for
the fixed effects coefficients $\beta$, and $Z_i$ a design matrix for the
random effects coefficients $b_i$.  Typically, matrix $Z_i$ is assumed to be a
subset of $X_i$.  The random effects are assumed to follow a normal distribution
with mean 0 and variance-covariance matrix $D$.  In addition, the distribution
$[y_i \mid b_i]$ may potentially have extra dispersion/shape parameters $\phi$.

### 1.2  Estimation

The package focuses on settings in which the distribution $[y_i \mid b_i]$ is
not normal and/or the link function $g(\cdot)$ is not the identity.  In these
settings, estimation is complicated because the marginal log-likelihood
$$\ell(\theta) = \sum_{i=1}^n \log \int p(y_i \mid b_i; \theta)\, p(b_i; \theta)\, db_i$$
does not have a closed-form solution.  This package provides an efficient
implementation of the **adaptive Gaussian quadrature** rule to approximate
these integrals accurately, combined with a hybrid EM + quasi-Newton
optimization scheme.

---

## 2  Generalized Linear Mixed Models — Practice

### 2.1  Mixed Effects Logistic Regression

We illustrate the use of the package with a mixed effects logistic regression.
We start by simulating some data for a binary longitudinal outcome.

In [ ]:
np.random.seed(1234)
n = 100   # number of subjects
K = 8     # number of measurements per subject
t_max = 15  # maximum follow-up time

# Construct a DataFrame with the design:
# everyone has a baseline measurement, then measurements at random follow-up times
ids = np.repeat(np.arange(1, n + 1), K)
times = np.concatenate([
    np.concatenate([[0], np.sort(np.random.uniform(0, t_max, K - 1))])
    for _ in range(n)
])
# gl(2, n/2, labels=c("male","female")) — first n/2 subjects male, next female
sex_labels = np.repeat(["male"] * (n // 2) + ["female"] * (n // 2), K)

DF = pd.DataFrame({"id": ids, "time": times, "sex": sex_labels})

# Design matrices for fixed (X) and random (Z) effects
DF["sex_female"] = (DF["sex"] == "female").astype(float)
X = np.column_stack([
    np.ones(n * K),
    DF["sex_female"],
    DF["time"],
    DF["sex_female"] * DF["time"],
])
Z = np.column_stack([np.ones(n * K), DF["time"]])

betas = np.array([-2.13, -0.25, 0.24, -0.05])
D11 = 0.48   # variance of random intercepts
D22 = 0.10   # variance of random slopes

# Simulate random effects
b = np.column_stack([
    np.random.normal(0, np.sqrt(D11), n),
    np.random.normal(0, np.sqrt(D22), n),
])
# Linear predictor
eta_y = X @ betas + np.sum(Z * b[DF["id"].values - 1, :], axis=1)
# Simulate binary longitudinal data
DF["y"] = np.random.binomial(1, expit(eta_y))

DF.head(10)

We fit a mixed effects logistic regression for `y`, assuming random intercepts
for the random-effects part.  The basic model-fitting class in **glmmadaptive**
is `MixedModel`, which mirrors R's `mixed_model()`.  It takes four required
arguments: `fixed`, `random`, `family`, and `data`.  The random effects formula
uses lme4-style `~ terms | group` syntax.

In [ ]:
fm1 = MixedModel(
    fixed="y ~ sex * time",
    random="~ 1 | id",
    family=Binomial(),
    data=DF,
).fit()

The `summary()` method gives a detailed output of the model:

In [ ]:
print(fm1.summary())

We continue by checking the impact of the chosen number of quadrature points on
the parameter estimates and the log-likelihood value at convergence.  The
default when the number of random effects is $\leq 2$ is 11 points
(`n_agh=11`).  We refit with 15 and 21 points by passing the `control`
dictionary:

In [ ]:
fm1_q11 = fm1  # already fitted with n_agh=11 (default)

fm1_q15 = MixedModel(
    fixed="y ~ sex * time", random="~ 1 | id",
    family=Binomial(), data=DF,
    control={"n_agh": 15},
).fit()

fm1_q21 = MixedModel(
    fixed="y ~ sex * time", random="~ 1 | id",
    family=Binomial(), data=DF,
    control={"n_agh": 21},
).fit()

models = {"nAGQ=11": fm1_q11, "nAGQ=15": fm1_q15, "nAGQ=21": fm1_q21}

We now extract from the models the estimated fixed effects (using `fixef()`),
the random-intercept variance, and the log-likelihood (`.logLik`):

In [ ]:
def extract(res):
    fe = res.fixef()
    return pd.Series(
        list(fe) + [res.D[0, 0], res.logLik],
        index=list(fe.index) + ["var_(Intercept)", "logLik"],
    )

comparison = pd.DataFrame({name: extract(res) for name, res in models.items()})
print(comparison.round(5).to_string())

We observe a rather stable model with virtually no differences between the
different choices of quadrature points.

We first compare the model with a simple logistic regression that does not
include any random effects (equivalent to R's `glm()`):

In [ ]:
# Fit null model (no random effects) — equivalent to R's glm(..., family=binomial())
km = smf.logit("y ~ sex * time", data=DF).fit(disp=False)

# Likelihood ratio test — 1 df: variance of the random intercept
lrt_stat = 2.0 * (fm1.logLik - km.llf)
p_value  = chi2.sf(lrt_stat, df=1)

print(f"LRT statistic : {lrt_stat:.4f}")
print(f"Degrees of freedom : 1")
print(f"p-value : {p_value:.4e}")

We obtain a highly significant p-value suggesting that there are correlations in
the data that cannot be ignored.  *Note:* the LRT calculates the p-value using
the standard $\chi^2$ distribution, here with one degree of freedom.  However,
because the null hypothesis for testing variance parameters is on the boundary
of the corresponding parameter space, it would be more appropriate to use a
mixture of $\chi^2$ distributions.

We extend model `fm1` by also including a random slopes term; however, we
assume that the covariance between the random intercepts and random slopes is
zero.  This is achieved by using the `||` symbol in the `random` argument
(equivalent to R's `~ time || id`):

In [ ]:
fm2 = MixedModel(
    fixed="y ~ sex * time",
    random="~ time || id",   # || forces a diagonal covariance matrix D
    family=Binomial(),
    data=DF,
).fit()

The likelihood ratio test between the two models is computed with
`MixModResults.anova()`.  The method automatically identifies which model has
the higher log-likelihood (the more complex model):

In [ ]:
print(MixModResults.anova(fm1, fm2))

The results suggest that we need the random slopes term.  We continue by testing
whether the covariance between the random effects terms is zero.  The model
under the alternative hypothesis uses the full (correlated) random effects
covariance matrix:

In [ ]:
fm3 = MixedModel(
    fixed="y ~ sex * time",
    random="~ time | id",    # full D: intercept + slope, with covariance estimated
    family=Binomial(),
    data=DF,
).fit()

In [ ]:
print(MixModResults.anova(fm2, fm3))

The results now suggest that the covariance between the two random effects
terms is not statistically different from zero.

---

### 2.2  Mixed Effects Poisson Regression

We continue our illustration with a Poisson mixed effects model, starting again
by simulating some longitudinal count data:

In [ ]:
np.random.seed(1234)
n = 100
K = 8
t_max = 15

ids = np.repeat(np.arange(1, n + 1), K)
times = np.concatenate([
    np.concatenate([[0], np.sort(np.random.uniform(0, t_max, K - 1))])
    for _ in range(n)
])
sex_labels = np.repeat(["male"] * (n // 2) + ["female"] * (n // 2), K)

DF2 = pd.DataFrame({"id": ids, "time": times, "sex": sex_labels})

DF2["sex_female"] = (DF2["sex"] == "female").astype(float)
X2 = np.column_stack([
    np.ones(n * K),
    DF2["sex_female"],
    DF2["time"],
    DF2["sex_female"] * DF2["time"],
])

betas2 = np.array([2.13, -0.25, 0.24, -0.05])
D11_p  = 0.48

b2 = np.random.normal(0, np.sqrt(D11_p), n)
eta_y2 = X2 @ betas2 + b2[DF2["id"].values - 1]
DF2["y"] = np.random.poisson(np.exp(eta_y2))

We fit the mixed effects Poisson regression for `y` assuming random intercepts:

In [ ]:
gm1 = MixedModel(
    fixed="y ~ sex * time",
    random="~ 1 | id",
    family=Poisson(),
    data=DF2,
).fit()

In [ ]:
print(gm1.summary())

In settings with few subjects, repeated measurements, or separation effects,
the package also allows placing a penalty on the fixed effects regression
coefficients $\beta$.  The penalty/prior is in the form of a Student's
$t$ distribution with mean 0, scale parameter 1, and 3 degrees of freedom,
placed on all $\beta$ coefficients except the intercept.

> **Note:** The `penalized` argument is not yet implemented in the Python port.
> It will be available in a future release.  In the R package the penalized
> model is fitted as:
>
> ```r
> gm2 <- mixed_model(y ~ sex * time, random = ~ 1 | id, data = DF,
>                    family = poisson(), penalized = TRUE)
> ```
>
> A ridge-like penalty can be specified by adjusting the degrees of freedom of
> the Student's t prior (e.g., `pen_df = 200` for a near-Gaussian ridge
> penalty).